# Kaggle YOLO11n training

Attach the private Kaggle dataset **`bdd100k-adpstack-subset`**, enable a **GPU accelerator** and **Internet**, and optionally add the Kaggle secret **`WANDB_API_KEY`**. This run uses a Tesla T4, YOLO11n, image size 640, batch 16, and 50 epochs. The measured one-epoch duration was 0.031 hours; that is timing information, not a final accuracy result. W&B is off by default; set `WANDB_ENABLED = True` below to use the secret.

In [ ]:
EPOCHS = 50
BATCH = 16
IMGSZ = 640
WANDB_ENABLED = False
WANDB_PROJECT = "adpstack"


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/30nikamvaishnavi/autonomous-driving-perception-stack.git"
REPO_DIR = Path("/kaggle/working/autonomous-driving-perception-stack")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,export]"],
    cwd=REPO_DIR,
    check=True,
)
sys.path.insert(0, str(REPO_DIR / "src"))
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable a Kaggle GPU accelerator, restart the session, and rerun.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

if WANDB_ENABLED:
    from kaggle_secrets import UserSecretsClient
    from ultralytics import settings

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    settings.update({"wandb": True})
else:
    from ultralytics import settings

    settings.update({"wandb": False})


In [ ]:
INPUT_DIR = Path("/kaggle/input")
LABEL_CANDIDATES = sorted(INPUT_DIR.rglob("det_train.json"))
BDD_ROOT = next(
    (
        path.parents[2]
        for path in LABEL_CANDIDATES
        if path.parent.name == "det_20"
        and path.parent.parent.name == "labels"
        and (path.parents[2] / "images" / "train").is_dir()
        and (path.parents[2] / "images" / "val").is_dir()
    ),
    None,
)
if BDD_ROOT is None:
    raise FileNotFoundError(
        "Could not find det_train.json and images/{train,val} under /kaggle/input; "
        "attach the bdd100k-adpstack-subset dataset."
    )
print(f"BDD100K subset: {BDD_ROOT}")


In [ ]:
from adpstack.data.bdd100k_to_yolo import convert_split, write_dataset_yaml

YOLO_DATA_DIR = Path("/kaggle/working/bdd_yolo")
for split, expected in (("train", 10_000), ("val", 2_000)):
    converted = convert_split(
        BDD_ROOT / "labels" / "det_20" / f"det_{split}.json",
        BDD_ROOT / "images" / split,
        YOLO_DATA_DIR,
        split,
        copy_images=True,
    )
    if converted != expected:
        raise RuntimeError(f"Expected {expected} {split} images, converted {converted}.")
YOLO_DATASET = write_dataset_yaml(YOLO_DATA_DIR)
print(f"YOLO dataset: {YOLO_DATASET}")


In [ ]:
from ultralytics import YOLO

RUNS_DIR = Path("/kaggle/working/runs/detect")
model = YOLO("yolo11n.pt")
model.train(
    data=str(YOLO_DATASET),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=4,
    seed=42,
    project=str(RUNS_DIR),
    name="yolo11n",
    cos_lr=True,
    patience=15,
    mosaic=1.0,
    mixup=0.1,
    plots=True,
)
BEST_PT = RUNS_DIR / "yolo11n" / "weights" / "best.pt"
if not BEST_PT.is_file():
    raise FileNotFoundError(f"Training checkpoint not found: {BEST_PT}")
print(f"Best checkpoint: {BEST_PT}")


In [ ]:
import subprocess

EVAL_PATH = Path("/kaggle/working/detection_eval.json")
subprocess.run(
    [
        sys.executable,
        "-m",
        "adpstack.detection.evaluate",
        "--weights",
        str(BEST_PT),
        "--data",
        str(YOLO_DATASET),
        "--labels-json",
        str(BDD_ROOT / "labels" / "det_20" / "det_val.json"),
        "--images-dir",
        str(YOLO_DATA_DIR / "images" / "val"),
        "--imgsz",
        str(IMGSZ),
        "--device",
        "0",
        "--out",
        str(EVAL_PATH),
    ],
    cwd=REPO_DIR,
    check=True,
)
print(f"Evaluation written to: {EVAL_PATH}")
